# RAG LLM Serving Smoke

Runs the LLM-enabled RAG path against the generation endpoint (default: Databricks FM API `databricks-meta-llama-3-3-70b-instruct`; External Model `access_drift_llm` also supported via widget). This is a serving smoke, not a full bronze/silver pipeline run.


In [ ]:
dbutils.widgets.text("catalog", "access_drift_dev")
dbutils.widgets.text("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.text("expected_min_rag_answers", "1")

CATALOG = dbutils.widgets.get("catalog")
LLM_ENDPOINT = dbutils.widgets.get("llm_endpoint")
EXPECTED_MIN_RAG_ANSWERS = int(dbutils.widgets.get("expected_min_rag_answers"))


In [ ]:
summary = spark.sql(f"""
SELECT
  answer_source,
  COUNT(*) AS cnt,
  SUM(CASE WHEN citations IS NULL OR citations = '[]' THEN 1 ELSE 0 END) AS citation_missing_cnt
FROM {CATALOG}.rag.recommended_action
GROUP BY answer_source
""")
display(summary)


In [ ]:
rag_answers = spark.sql(f"""
SELECT COUNT(*) AS cnt
FROM {CATALOG}.rag.recommended_action
WHERE answer_source = 'rag'
  AND llm_endpoint = '{LLM_ENDPOINT}'
  AND citations IS NOT NULL
  AND citations <> '[]'
""").collect()[0]["cnt"]

assert rag_answers >= EXPECTED_MIN_RAG_ANSWERS, (
    f"Expected at least {EXPECTED_MIN_RAG_ANSWERS} LLM-backed RAG answers with citations; got {rag_answers}. "
    "Check the generation endpoint and rerun rag_chain_task."
)
print(f"LLM serving smoke passed: rag_answers={rag_answers}")
